# Module 17 — Local Coding Assistants

Companion notebook to [`docs/modules/17_local_coding_assistants.md`](../docs/modules/17_local_coding_assistants.md).

The capstone of the Agents/tools phase, wiring together Modules 14-16: real AST-based repo
indexing, a real proposed-and-applied patch fixing a genuine pre-existing bug in
`datasets/code_repos/mini_calculator/`, and a real pytest run proving the fix — failure before,
pass after. Every stage runs for real except the one LLM call proposing the patch text
(`FakeRuntime`-backed).


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_17"))
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. The sample repo's real, pre-existing bug

In [2]:
SAMPLE_REPO = REPO_ROOT / "datasets" / "code_repos" / "mini_calculator"
print((SAMPLE_REPO / "calculator.py").read_text())


"""A tiny calculator module - Module 17's sample repo for the coding assistant labs."""


def add(a: float, b: float) -> float:
    return a + b


def subtract(a: float, b: float) -> float:
    return a - b


def multiply(a: float, b: float) -> float:
    return a * b


def divide(a: float, b: float) -> float:
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


def average(numbers: list[float]) -> float:
    return sum(numbers) / len(numbers)



## 2. Labs 1-2: index the repo, real AST-based symbols, architecture questions

In [3]:
from index_repo_demo import run_lab as run_index_lab, result_to_markdown as index_to_markdown

index_result = await run_index_lab()
print(index_to_markdown(index_result))


# Labs 1-2 - repo index and architecture questions

- Files indexed: ['calculator.py', 'tests/test_calculator.py']

## Repo map (real AST symbols)

- calculator.py:
  - function add (line 4)
  - function subtract (line 8)
  - function multiply (line 12)
  - function divide (line 16)
  - function average (line 22)
- tests/test_calculator.py:
  - function test_add (line 6)
  - function test_subtract (line 10)
  - function test_multiply (line 14)
  - function test_divide (line 18)
  - function test_divide_by_zero_raises (line 22)
  - function test_average_of_nonempty_list (line 27)
  - function test_average_of_empty_list_should_return_zero (line 31)

## "What functions exist in calculator.py?"
['add', 'subtract', 'multiply', 'divide', 'average']


## search_repo('ValueError')
['calculator.py:18', 'tests/test_calculator.py:23']




## 3. Lab 3: generate a test for one function, then actually run it

In [4]:
from generate_tests_demo import run_lab as run_generate_lab, result_to_markdown as generate_to_markdown

generate_result = await run_generate_lab()
print(generate_to_markdown(generate_result))


# Lab 3 - generate a test for one function, then actually run it

- Generated test source:
```python
def test_subtract_generated():
    from calculator import subtract
    assert subtract(10, 4) == 6
```
- Passing test count before adding the generated test: 6
- Passing test count after adding the generated test: 7 (the new test genuinely runs and passes; the repo's one pre-existing unrelated failure is untouched)
- Final pytest summary line: 1 failed, 7 passed in 0.02s



## 4. Labs 4-7: propose, validate, approve, apply a real patch — real tests before and after

The strongest "real proof" this course has produced: an actual pytest failure, an actual patch
applied to an actual file, and an actual pytest pass — not staged, not simulated.

In [5]:
from patch_and_test_demo import run_lab as run_patch_lab, result_to_markdown as patch_to_markdown

patch_result = await run_patch_lab()
print(patch_to_markdown(patch_result))


# Labs 4-7 - propose, validate, approve, apply a real patch; run real tests before and after

- **Before the patch**: tests passed = False -> `1 failed, 6 passed in 0.02s`
- Apply denied with no real approval gate: approval_denied
- Apply approved with a real approval gate: end
- **After the patch**: tests passed = True -> `7 passed in 0.01s`
- Hallucinated patch rejected (context mismatch): True
- File untouched after the rejected hallucinated patch: True



## 5. Code hallucination made measurable

A patch describing code that doesn't actually exist in the file (`statistics.mean(numbers)`
instead of the real `sum(numbers) / len(numbers)`) is rejected by a real context-match check,
not misapplied.

In [6]:
print("Hallucinated patch rejected:", patch_result["hallucinated_patch_rejected"])
print("File left untouched:", patch_result["hallucinated_patch_file_untouched"])


Hallucinated patch rejected: True
File left untouched: True
